In [19]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [38]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

In [3]:
from summft.data import len_stats

len_stats()

--------------dialogue--------------
mean = 140.8 
median = 114.0
95th = 342.0
99th = 490.0
max = 1049
--------------summary--------------
mean = 25.8 
median = 23.0
95th = 54.0
99th = 67.0
max = 86
Number of samples: 14731


In [39]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [40]:
dataset = load_dataset("knkarthick/samsum")
example = dataset["train"][0]

In [88]:
def format_example(example):
    messages = (
        {
            "role": "system",
            "content": "Summarize this dialogue in one or two sentences.",
        },
        {"role": "user", "content": example["dialogue"]},
        {"role": "assistant", "content": example["summary"]},
    )

    return messages


systems, prompts, answers = [], [], []
for i in range(7):
    system, prompt, answer = format_example(dataset["train"][i])
    systems.append(system)
    prompts.append(prompt)
    answers.append(answer)
for i in range(4):
    temp_list = [systems[i]]
    # for j in (4, 5, 6):
    #    temp_list.extend([prompts[j], answers[j]])
    temp_list.append(prompts[i])
    prompts[i] = temp_list
for i in range(4):
    print(f"---------Example {i + 1}---------")
    for turn in prompts[i]:
        print(turn["content"])
    print()
    print(answers[i]["content"])
    print()

---------Example 1---------
Summarize this dialogue in one or two sentences.
Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)

Amanda baked cookies and will bring Jerry some tomorrow.

---------Example 2---------
Summarize this dialogue in one or two sentences.
Olivia: Who are you voting for in this election? 
Oliver: Liberals as always.
Olivia: Me too!!
Oliver: Great

Olivia and Olivier are voting for liberals in this election. 

---------Example 3---------
Summarize this dialogue in one or two sentences.
Tim: Hi, what's up?
Kim: Bad mood tbh, I was going to do lots of stuff but ended up procrastinating
Tim: What did you plan on doing?
Kim: Oh you know, uni stuff and unfucking my room
Kim: Maybe tomorrow I'll move my ass and do everything
Kim: We were going to defrost a fridge so instead of shopping I'll eat some defrosted veggies
Tim: For doing stuff I recommend Pomodoro technique where u use breaks for doing chores
Tim: It really helps
Kim

In [14]:
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 5960.94it/s]


In [ ]:
responses = []
for prompt in prompts[:4]:
    inputs = tokenizer.apply_chat_template(
        [prompt],
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    out = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    new_tokens = out[0][inputs["input_ids"].shape[1] :]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    responses.append(response)
print(f"Total of {len(responses)} responses.")

Total of 4 responses.


In [90]:
for response, answer in zip(responses, answers):
    print("Decoder summary:")
    print(response)
    print("\nHuman summary:")
    print(answer["content"])
    print("-" * 10)

Decoder summary:
Amanda and Jerry are planning to bake cookies together for a friend's birthday party on the following day.

Human summary:
Amanda baked cookies and will bring Jerry some tomorrow.
----------
Decoder summary:
Olivia and Oliver are both voting for the same political party, which is the Liberal Party.

Human summary:
Olivia and Olivier are voting for liberals in this election. 
----------
Decoder summary:
Tim asks Kim about her current mood and plans, while Kim mentions she was planning to do various tasks but ended up procrastinating. Tim suggests the Pomodoro Technique as an effective way to manage time more efficiently.

Human summary:
Kim may try the pomodoro technique recommended by Tim to get more stuff done.
----------
Decoder summary:
Rachel is outside and Edward thinks he's in over his head.

Human summary:
Edward thinks he is in love with Bella. Rachel wants Edward to open his door. Rachel is outside. 
----------


In [9]:
import json
from pathlib import Path
import pandas as pd

rows = []
for p in Path("results").glob("icl_*_validation/metrics.json"):
    rows.append(json.load(open(p)))
table = pd.DataFrame(rows).sort_values("shots")
table[["rouge1", "rouge2", "rougeL"]] = (
    table[["rouge1", "rouge2", "rougeL"]] * 100
).round(1)
table = table[["shots", "rouge1", "rouge2", "rougeL", "minutes", "n"]]
table.style.hide(axis="index").format(precision=1)
# table

shots,rouge1,rouge2,rougeL,minutes,n
0,33.8,10.0,25.2,18.4,818
1,35.0,11.4,27.7,10.9,818
3,38.4,13.2,29.7,15.9,818
5,38.9,13.8,29.8,18.0,818


In [10]:
print(table.to_string(index=False))

 shots  rouge1  rouge2  rougeL  minutes   n
     0    33.8    10.0    25.2     18.4 818
     1    35.0    11.4    27.7     10.9 818
     3    38.4    13.2    29.7     15.9 818
     5    38.9    13.8    29.8     18.0 818


In [14]:
import pandas as pd

for k in [0, 1, 3, 5]:
    df = pd.read_json(f"results/icl_{k}shot_validation/predictions.jsonl", lines=True)
    pred_len = df["prediction"].str.split().str.len().mean()
    ref_len = df["reference"].str.split().str.len().mean()
    print(f"{k} shots: pred len = {pred_len:.1f} ref len = {ref_len:.1f}")

0 shots: pred len = 33.2 ref len = 20.3
1 shots: pred len = 18.5 ref len = 20.3
3 shots: pred len = 26.3 ref len = 20.3
5 shots: pred len = 29.3 ref len = 20.3


In [18]:
runs = {}
for k in [0, 1, 3, 5]:
    runs[k] = pd.read_json(
        f"results/icl_{k}shot_validation/predictions.jsonl", lines=True
    )

for i in range(8):
    print(f"--- {runs[0]['id'][i]} ---")
    print("REF: ", runs[0]["reference"][i])
    for k in [0, 1, 3, 5]:
        print(f"{k}-shot: " + runs[k]["prediction"][i])

--- 13817023 ---
REF:  A will go to the animal shelter tomorrow to get a puppy for her son. They already visited the shelter last Monday and the son chose the puppy. 
0-shot: Tom asks A if he can go to the animal shelter with him and gets a puppy for his son.
1-shot: Tom asks A if he can go to the animal shelter with him.
3-shot: Tom and A plan to visit the animal shelter together.
5-shot: Tom is going to the animal shelter with A.
--- 13716628 ---
REF:  Emma and Rob love the advent calendar. Lauren fits inside calendar various items, for instance, small toys and Christmas decorations. Her children are excited whenever they get the calendar.
0-shot: Emma and Lauren both express their excitement at finding an advent calendar for their children. Emma mentions she loves the idea of filling the calendar with different items, while Lauren suggests adding notes from her sister to make it more meaningful. Both agree that the advent calendars help prepare children for Christmas traditions.
1-s

In [27]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from summft.mmlu import LETTERS, format_question, mmlu_correct
import torch

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

questions = {
    "question": "What is 2 + 2?",
    "choices": ["2", "4", "88", "97"],
    "answer": 1,
}
inputs = tokenizer.apply_chat_template(
    [{"role": "user", "content": format_question(questions)}],
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
)
letter_ids = [tokenizer.encode(x, add_special_tokens=False)[0] for x in LETTERS]
model.eval()

with torch.no_grad():
    logits = model(**inputs).logits[0, -1]
print(dict(zip(LETTERS, logits[letter_ids].softmax(0))))

output = model.generate(**inputs, max_new_tokens=10, do_sample=False)
print(repr(tokenizer.decode(output[0, inputs["input_ids"].shape[1] :])))

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 5752.06it/s]


{'A': tensor(0.7695, dtype=torch.bfloat16), 'B': tensor(0.2207, dtype=torch.bfloat16), 'C': tensor(0.0032, dtype=torch.bfloat16), 'D': tensor(0.0046, dtype=torch.bfloat16)}
'A. 2\n\nThe sum of 2'
